In [2]:
import torch 
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torchvision import models 
from torchvision.models import VGG16_Weights

In [ ]:
class SSD(nn.Module):
    def __init__(self,num_classes):
        super(SSD,self).__init__()
        self.num_classes=num_classes 
        vgg=models.vgg16(weights=VGG16_Weights.IMAGENET1K_V1).features
        self.features=nn.ModuleList(vgg[:30])

        self.extras=nn.ModuleList([
            nn.Sequential(
                nn.Conv2d(512,1024,kernel_size=3,padding=1,dilation=1),
                nn.ReLU(inplace=True)
            ),
            nn.Sequential(
                nn.Conv2d(1024,256,kernel_size=1),
                nn.ReLU(inplace=True),
                nn.Conv2d(256,512,kernel_size=3,padding=1,dilation=1),
                nn.ReLU(inplace=True)
            ),
            nn.Sequential(
                nn.Conv2d(512,128,kernel_size=1),
                nn.ReLU(inplace=True),
                nn.Conv2d(128,256,kernel_size=3,padding=1,dilation=1),
                nn.ReLU(inplace=True)
            ),
            nn.Sequential(
                nn.Conv2d(256,128,kernel_size=1),
                nn.ReLU(inplace=True),
                nn.Conv2d(128,256,kernel_size=3,padding=1,dilation=1),
                nn.ReLU(inplace=True)
            ),
            nn.Sequential(
                nn.Conv2d(256,128,kernel_size=1),
                nn.ReLU(inplace=True),
                nn.Conv2d(128,256,kernel_size=3,padding=1,dilation=1),
                nn.ReLU(inplace=True)
            )
        ])
        self.loc=nn.ModuleList([
            nn.Conv2d(512,4*4,kernel_size=3,padding=1),
            nn.Conv2d(1024,6*4,kernel_size=3,padding=1),
            nn.Conv2d(512,6*4,kernel_size=3,padding=1),
            nn.Conv2d(256,6*4,kernel_size=3,padding=1),
            nn.Conv2d(256,4*4,kernel_size=3,padding=1),
            nn.Conv2d(256,4*4,kernel_size=3,padding=1)
        ])
        self.conf=nn.ModuleList([
            nn.Conv2d(512,self.num_classes*4,kernel_size=3,padding=1),
            nn.Conv2d(1024,self.num_classes*6,kernel_size=3,padding=1),
            nn.Conv2d(512,self.num_classes*6,kernel_size=3,padding=1),
            nn.Conv2d(256,self.num_classes*6,kernel_size=3,padding=1),
            nn.Conv2d(256,self.num_classes*4,kernel_size=3,padding=1),
            nn.Conv2d(256,self.num_classes*4,kernel_size=3,padding=1)
        ])
    def forward(self,x):
        locs=[]
        confs=[]
        for k in range(len(self.features)):
            x=self.features[k](x)
        locs.append(self.loc[0](x).permute(0,2,3,1).contiguous())
        confs.append(self.conf[0](x).permute(0,2,3,1).contiguous())
        for k in range(len(self.extras)):
            x=self.extras[k](x)
            locs.append(self.loc[k+1](x).permute(0,2,3,1).contiguous())
            confs.append(self.conf[k+1](x).permute(0,2,3,1).contiguous())
        locs=torch.cat([o.view(o.size(0),-1) for o in locs],dim=1)
        confs=torch.cat([o.view(o.size(0),-1) for o in confs],dim=1)
        return locs.view(locs.size(0),-1,4),confs.view(confs.size(0),-1,self.num_classes)   
if __name__=="__main__":
    model=SSD(num_classes=21)
    x=torch.randn(1,3,300,300)
    locs,confs=model(x)
    print(locs.shape)
    print(confs.shape)